In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns

In [ ]:
def load_celltype_results(input_dir: Path):
    """Load saved pseudobulk matrices and metadata for each cell type."""
    results = []
    for cell_type_dir in sorted(p for p in input_dir.iterdir() if p.is_dir()):
        matrix_files = list(cell_type_dir.glob("*_pseudobulk_matrix_ROI.csv"))
        meta_files = list(cell_type_dir.glob("*_pseudobulk_metadata_ROI.csv"))

        if not matrix_files or not meta_files:
            continue

        matrix_file = matrix_files[0]
        meta_file = meta_files[0]
        cell_type_label = matrix_file.name.replace("_pseudobulk_matrix_ROI.csv", "")

        pb_sample = pd.read_csv(matrix_file, index_col=0)
        meta_df = pd.read_csv(meta_file, index_col=0)
        meta_df.index = meta_df.index.astype(str)

        common_samples = [
            sample for sample in pb_sample.columns if sample in meta_df.index
        ]
        if not common_samples:
            continue

        pb_sample = pb_sample[common_samples]
        meta_df = meta_df.loc[common_samples]

        results.append((cell_type_label, cell_type_dir, pb_sample, meta_df))
    return results


def plot_metric(df, x, y, cell_type, palette):
    """Plot a metric (y) by a grouping variable (x) for a given cell type."""
    plot_df = df[[x, y]].dropna().copy()
    plot_df[x] = plot_df[x].astype(str)

    fig, ax = plt.subplots(figsize=(6, 4))
    group_order = list(pd.unique(plot_df[x]))
    # Use the provided palette dict; fall back to a grey for unknown categories
    palette_to_use = [palette.get(g, "#888888") for g in group_order]

    sns.boxplot(
        data=plot_df,
        x=x,
        y=y,
        order=group_order,
        hue=x,
        palette=palette_to_use,
        saturation=0.75,
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x=x,
        y=y,
        hue=x,
        order=group_order,
        palette=palette_to_use,
        size=2.5,
        jitter=True,
        ax=ax,
        legend=False,
    )

    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f"{cell_type}: {y} by {x}")
    plt.tight_layout()

    return fig

In [4]:
"""Visualize number of cells per sample."""
# Set directory
path = Path(
    "/Volumes/phenotypingsputumasthmaticsaurorawellcomea1/live/Sara_Patti/009_ST_Xenium"
)

# Set directories
input_dir = path / "output" / "pb" / "pb_data_celltype"
out_dir = path / "output" / "pb" / "num_cells_IPF"
out_dir.mkdir(parents=True, exist_ok=True)

# set fig dir for plots to save to
sc.settings.figdir = out_dir


results = load_celltype_results(input_dir)

# Load total number of cells per sample
total_cells = pd.read_csv(
    path / "output" / "pb" / "pb_data_concatenated" / "pseudobulk_metadata_ROI.csv",
    index_col=0,
)

In [27]:
total_cells

,n_cells,total_counts,mean_transcripts,sample_ID,batch,condition,timepoint,timepoint_label,sample_ID_manual,time_point,...,ppFEV1,ppFVC,FEV1_FVC,ppRV,ppTLC,RV_TLC,TLCO,ppTLCO,KCO,KCO_percent
ROI,,,,,,,,,,,,,,,,,,,,,
COPD_46005_V1,23129,1739235.0,75.197155,COPD_46005,4,COPD,V1,baseline,COPD_46005,V1,...,0.360,1.000,0.2800,1.98,1.41,0.5723,NaN,NaN,NaN,NaN
COPD_46005_V2,25141,1894234.0,75.344417,COPD_46005,4,COPD,V2,6_weeks,COPD_46005,V2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
COPD_R003_V1,17710,860335.0,48.579051,COPD_R003,2,COPD,V1,baseline,COPD_R003,V1,...,0.300,0.700,0.2852,1.55,0.99,0.5466,NaN,NaN,NaN,NaN
COPD_R003_V2,33066,5211067.0,157.595929,COPD_R003,3,COPD,V2,6_weeks,COPD_R003,V2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
COPD_R003_V3,4841,388271.0,80.204710,COPD_R003,1,COPD,V3,6_months,COPD_R003,V3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
COPD_R009_V1,12921,1404592.0,108.706137,COPD_R009,2,COPD,V1,baseline,COPD_R009,V1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
COPD_R009_V2,38399,4851330.0,126.340009,COPD_R009,2,COPD,V2,6_weeks,COPD_R009,V2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
COPD_R009_V3,36978,5699096.0,154.121261,COPD_R009,1,COPD,V3,6_months,COPD_R009,V3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
COPD_R010_V2,25500,3743171.0,146.791020,COPD_R010,1,COPD,V2,6_weeks,COPD_R010,V1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
test = results[0]
test[2].head()

,IPF_RBH_01,IPF_RBH_02,IPF_RBH_03,IPF_RBH_04,IPF_RBH_06,IPF_RBH_14,IPF_RBH_15_CORRECT,IPF_RBH_15_OG,IPF_RBH_16,IPF_RBH_18,IPF_RBH_19,PM08_159,PM08_162,PM08_163,PM08_164,PM08_167,PM08_169
16S,177.0,8.0,321.0,317.0,218.0,37.0,58.0,114.0,261.0,7.0,27.0,785.0,736.0,71.0,101.0,219.0,44.0
A2ML1,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAMP,57.0,1.0,65.0,67.0,20.0,7.0,11.0,31.0,57.0,3.0,5.0,175.0,40.0,36.0,80.0,81.0,11.0
AAR2,45.0,1.0,27.0,32.0,10.0,2.0,5.0,12.0,22.0,0.0,3.0,52.0,6.0,10.0,21.0,20.0,2.0
AARSD1,26.0,0.0,12.0,16.0,9.0,4.0,3.0,4.0,13.0,0.0,1.0,58.0,12.0,13.0,13.0,22.0,6.0


In [25]:
total_cells.loc["IPF_RBH_16"]

n_cells                                                    36999
total_counts                                           8984506.0
mean_transcripts                                      242.831049
sample_ID                                             IPF_RBH_16
batch                                                          1
condition                                                    IPF
timepoint                                                    NaN
timepoint_label                                              NaN
sample_ID_manual                                      IPF_RBH_16
time_point                                                   NaN
time_point_label                                             NaN
lung_location                                             DISTAL
biopsy_type                                TRANSBRONCHIAL BIOPSY
ID                                                           NaN
study                                                        RBH
diagnosis                

In [26]:
meta_test.loc["IPF_RBH_16"]

n_cells                                                      822
total_counts                                            212170.0
mean_transcripts                                      258.114355
sample_ID                                             IPF_RBH_16
batch                                                          1
condition                                                    IPF
timepoint                                                    NaN
timepoint_label                                              NaN
sample_ID_manual                                      IPF_RBH_16
time_point                                                   NaN
time_point_label                                             NaN
lung_location                                             DISTAL
biopsy_type                                TRANSBRONCHIAL BIOPSY
ID                                                           NaN
study                                                        RBH
diagnosis                

In [2]:
cmap_blues = sns.color_palette("ch:start=.2,rot=-.3", as_cmap=True)

In [3]:
adata = sc.read_h5ad(
    "/Volumes/phenotypingsputumasthmaticsaurorawellcomea1/live/Sara_Patti/009_ST_Xenium/COPD_V1_v_IPF/annotate/adata.h5ad"
)

In [4]:
annotation_var = "manual_annotation"
adata.obs[annotation_var].unique().tolist()

['Basal epithelial cells',
 'Goblet cells MUC5AChi',
 'Goblet cells MUC5Bhi',
 'Ciliated epithelial cells',
 'Fibroblast',
 'T cells',
 'Macrophages',
 'Lymphatic endothelial cells',
 'Blood endothelial cells',
 'Alveolar epithelial cells',
 'Serous acinar cells',
 'Plasma cells',
 'Mast cells',
 'Endothelial cells',
 'Pericytes',
 'Smooth muscle cells']

In [ ]:
# Epithelial cells
epi_types = [
    "Alveolar epithelial cells",
    "Basal epithelial cells",
    "Goblet cells MUC5AChi",
    "Goblet cells MUC5Bhi",
    "Ciliated epithelial cells",
    "Serous acinar cells",
]
adata_sub = adata[adata.obs[annotation_var].isin(epi_types), :].copy()
adata_sub.write_h5ad(
    "/Volumes/phenotypingsputumasthmaticsaurorawellcomea1/live/Sara_Patti/009_ST_Xenium/COPD_V1_v_IPF/manual/subset/adata_epi.h5ad"
)

RuntimeError: Disable slist on flush dest failure failed (file write failed: time = Tue Jan  6 18:55:36 2026
, filename = '/Volumes/phenotypingsputumasthmaticsaurorawellcomea1/live/Sara_Patti/009_ST_Xenium/COPD_V1_v_IPF/manual/subset/adata_epi.h5ad', file descriptor = 74, errno = 5, error message = 'Input/output error', buf = 0x7fea431f7a00, total write size = 4096, bytes this sub-write = 4096, offset = 2048)

In [5]:
immune_types = [
    "T cells",
    "Plasma cells",
    "Macrophages",
    "Mast cells",
]

adata_immune = adata[adata.obs[annotation_var].isin(immune_types), :].copy()

# adata_immune.write_h5ad(
#     "/Volumes/phenotypingsputumasthmaticsaurorawellcomea1/live/Sara_Patti/009_ST_Xenium/COPD_V1_v_IPF/manual/subset/adata_immune.h5ad"
# )

In [ ]:
vascular_types = [
    "Lymphatic endothelial cells",
    "Blood endothelial cells",
    "Endothelial cells",
]

adata_vascular = adata[adata.obs[annotation_var].isin(vascular_types), :].copy()

adata_vascular.write_h5ad(
    "/Volumes/phenotypingsputumasthmaticsaurorawellcomea1/live/Sara_Patti/009_ST_Xenium/COPD_V1_v_IPF/manual/subset/adata_vascular.h5ad"
)

In [ ]:
stromal_types = [
    "Fibroblast",
    "Smooth muscle cells",
    "Pericytes",
]

adata_stromal = adata[adata.obs[annotation_var].isin(stromal_types), :].copy()

adata_stromal.write_h5ad(
    "/Volumes/phenotypingsputumasthmaticsaurorawellcomea1/live/Sara_Patti/009_ST_Xenium/COPD_V1_v_IPF/manual/subset/adata_stromal.h5ad"
)

In [ ]:
adata_sub = adata[adata.obs[annotation_var] == "Fibroblast", :].copy()
adata_sub.write_h5ad(
    "/Volumes/phenotypingsputumasthmaticsaurorawellcomea1/live/Sara_Patti/009_ST_Xenium/COPD_V1_v_IPF/manual/subset/adata_fibroblast.h5ad"
)

adata_sub = adata[adata.obs[annotation_var] == "T cells", :].copy()
adata_sub.write_h5ad(
    "/Volumes/phenotypingsputumasthmaticsaurorawellcomea1/live/Sara_Patti/009_ST_Xenium/COPD_V1_v_IPF/manual/subset/adata_Tcells.h5ad"
)

adata_sub = adata[adata.obs[annotation_var] == "Macrophages", :].copy()
adata_sub.write_h5ad(
    "/Volumes/phenotypingsputumasthmaticsaurorawellcomea1/live/Sara_Patti/009_ST_Xenium/COPD_V1_v_IPF/manual/subset/adata_macrophages.h5ad"
)

adata_sub = adata[adata.obs[annotation_var] == "Alveolar epithelial cells", :].copy()
adata_sub.write_h5ad(
    "/Volumes/phenotypingsputumasthmaticsaurorawellcomea1/live/Sara_Patti/009_ST_Xenium/COPD_V1_v_IPF/manual/subset/adata_at1_at2.h5ad"
)